# 🧠 Model Training — EfficientNetB0 Transfer Learning
**RoadSense AI | Training Notebook**

This notebook covers:
- Data generator setup
- Phase 1: Feature extraction (frozen backbone)
- Phase 2: Fine-tuning (top layers unfrozen)
- Training curve visualization
- Model saving

In [ ]:
import os, sys
sys.path.insert(0, '..')
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import yaml, pickle

from src.data_preparation import load_dataset, create_data_generators
from src.model import build_model, unfreeze_base
from src.utils import plot_training_history
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

with open('../config.yaml') as f:
    config = yaml.safe_load(f)

print('TensorFlow version:', tf.__version__)
print('GPU available:', len(tf.config.list_physical_devices('GPU')) > 0)

## 1. Load Dataset & Create Generators

In [ ]:
image_paths, labels = load_dataset(config['data']['dataset_path'])
print(f'Total images: {len(image_paths)}')

from collections import Counter
print('Class distribution:', Counter(labels))

train_gen, val_gen = create_data_generators(
    image_paths, labels,
    batch_size=config['data']['batch_size'],
    img_size=tuple(config['data']['image_size']),
    validation_split=config['data']['validation_split'],
    seed=config['data']['seed']
)
print(f'Train batches: {len(train_gen)} | Val batches: {len(val_gen)}')

## 2. Build Model

In [ ]:
model = build_model(
    num_classes=len(config['classes']),
    input_shape=(*config['data']['image_size'], 3),
    dropout_rate=config['model']['dropout_rate'],
    learning_rate=config['model']['learning_rate']
)
model.summary()

## 3. Phase 1 — Feature Extraction (Epochs 1–10)

In [ ]:
os.makedirs('../models', exist_ok=True)
callbacks = [
    ModelCheckpoint('../models/damage_classifier.h5', monitor='val_accuracy',
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6, verbose=1)
]

history1 = model.fit(
    train_gen, validation_data=val_gen,
    epochs=10, callbacks=callbacks, verbose=1
)
print('Phase 1 complete. Best val_accuracy:', max(history1.history['val_accuracy']))

## 4. Phase 2 — Fine-Tuning (Epochs 11–30)

In [ ]:
model = unfreeze_base(model, layers_to_unfreeze=20)
history2 = model.fit(
    train_gen, validation_data=val_gen,
    epochs=30, initial_epoch=10,
    callbacks=callbacks, verbose=1
)
print('Phase 2 complete. Best val_accuracy:', max(history2.history['val_accuracy']))

## 5. Training Curves

In [ ]:
# Merge histories
full_history = {}
for key in history1.history:
    full_history[key] = history1.history[key] + history2.history[key]

plot_training_history(full_history, save_dir='../reports/figures')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(full_history['accuracy'],     label='Train', color='#4a90d9')
axes[0].plot(full_history['val_accuracy'], label='Val',   color='#e94560', linestyle='--')
axes[0].axvline(x=10, color='gray', linestyle=':', label='Fine-tuning start')
axes[0].set_title('Accuracy'); axes[0].legend()

axes[1].plot(full_history['loss'],     label='Train', color='#48bb78')
axes[1].plot(full_history['val_loss'], label='Val',   color='#f6ad55', linestyle='--')
axes[1].axvline(x=10, color='gray', linestyle=':')
axes[1].set_title('Loss'); axes[1].legend()
plt.tight_layout(); plt.show()

## 6. Save Class Names

In [ ]:
class_names = {v: k for k, v in config['classes'].items()}
with open('../models/class_names.pkl', 'wb') as f:
    pickle.dump(class_names, f)
print('Saved class_names.pkl')
print('Class mapping:', class_names)